# Eksperimen adaptasi aktivitas manusia

Jalankan dari atas pada kernel HumanActivity GPU. Enam kelas KTH ditambah idle. Video lokal hanya untuk training; belum ada test independen video pribadi atau idle. Rekaman jogging lokal boleh kosong.

Perubahan: jendela dua detik untuk training/inferensi, letterbox untuk menjaga proporsi, dan keluaran terpisah di outputs_adaptasi. Model lama tetap tersedia. Penambahan sedikit data belum menjamin perbaikan.


In [ ]:
import os
os.environ['KERAS_BACKEND']='torch'
from pathlib import Path
ROOT=Path.cwd().resolve()
if not (ROOT/'Pengenalan_Aktivitas_KTH_Local.ipynb').exists():
    raise RuntimeError('Jalankan Jupyter dari folder notebooks/ repository ini.')
ZIP_DIR=Path(os.environ.get('KTH_DATASET_DIR', str(ROOT/'datasets'/'KTH')))
OUT=ROOT/'outputs_adaptasi'
CACHE=ROOT/'cache_adaptasi'
LOCAL=ROOT/'runtime'/'data'
for p in (OUT,CACHE,LOCAL,ROOT/'inference'): p.mkdir(parents=True,exist_ok=True)
SEED=42
FRAMES=16
IMAGE_SIZE=160
BATCH_SIZE=32
EPOCHS=40
INFERENCE_VIDEO=ROOT/'inference'/'video_uji.mp4'
WINDOW_SECONDS=2.0
PREDICT_EVERY_SECONDS=0.5
KTH_CLASSES=['boxing','handclapping','handwaving','jogging','running','walking']
CLASSES=KTH_CLASSES+['idle']
print('Dataset:',ZIP_DIR)
print('Keluaran:',OUT)


In [ ]:
import os, re, json, zipfile, shutil, hashlib, subprocess, importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import torch
import keras
import requests
from collections import deque
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score, f1_score
keras.utils.set_random_seed(SEED)
print('Keras',keras.__version__,'backend',keras.backend.backend(),'PyTorch',torch.__version__,'OpenCV',cv2.__version__)
torch.set_num_threads(8)
print('CUDA:',torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('GPU CUDA tidak tersedia; jalankan check_setup.py.')
print('GPU:',torch.cuda.get_device_name(0))


## Ekstrak dan validasi dataset
Arsip dibaca dari folder lokal, lalu diekstrak ke runtime/data. Direktori dataset lama kantuk tidak dipakai. Ekstraksi menolak path arsip yang keluar dari direktori tujuan.

In [ ]:
for action in KTH_CLASSES:
    archive = ZIP_DIR / f'{action}.zip'
    if not archive.exists():
        raise FileNotFoundError(f'Belum ada: {archive}. Unggah ZIP terlebih dahulu.')
    destination = LOCAL / action
    marker = destination / '.complete'
    signature = f'{archive.stat().st_size}:{archive.stat().st_mtime_ns}'
    if marker.exists() and marker.read_text() == signature:
        continue
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination.resolve()):
                raise ValueError('Path ZIP tidak aman')
        z.extractall(destination)
    marker.write_text(signature)

TRAIN_IDS = {11,12,13,14,15,16,17,18}
VAL_IDS = {19,20,21,23,24,25,1,4}
TEST_IDS = {22,2,3,5,6,7,8,9,10}
assert not (TRAIN_IDS & VAL_IDS or TRAIN_IDS & TEST_IDS or VAL_IDS & TEST_IDS)
rows = []
for action in KTH_CLASSES:
    for p in sorted((LOCAL/action).rglob('*.avi')):
        m = re.fullmatch(r'person(\d+)_(\w+)_d([1-4])(?:_uncomp)?',p.stem)
        if not m or m[2] != action:
            raise ValueError(f'Nama video tidak dikenal: {p.name}')
        person = int(m[1])
        split = 'train' if person in TRAIN_IDS else 'val' if person in VAL_IDS else 'test' if person in TEST_IDS else None
        if split is None: raise ValueError('Subjek tidak dikenal')
        rows.append(dict(path=str(p),video_id=f'person{person:02d}_{action}_d{m[3]}',subject_id=person,action=action,label=CLASSES.index(action),split=split))
videos = pd.DataFrame(rows)
if videos.empty or videos.video_id.duplicated().any():
    raise ValueError('Dataset kosong atau video duplikat')
display(pd.crosstab(videos.action,videos.split))
for split in ['train','val','test']:
    assert videos[videos.split==split].label.nunique()==len(KTH_CLASSES)
videos.to_csv(OUT/'video_manifest.csv',index=False)
print('Total video:',len(videos), '(arsip KTH ini berisi 599 video)')


## Persiapan jendela dua detik dan video tambahan
Eksperimen ini memakai jendela tetap dalam video, bukan segmentasi subsekuens resmi KTH. Split orang KTH tetap mengikuti protokol resmi. Semua potongan video tambahan tetap dalam train.


In [ ]:
# Semua rekaman tambahan dipakai untuk adaptasi train saja.
# Prefix local_ mencegah benturan ID dengan orang dalam dataset KTH.
extra=[]
for action in CLASSES:
    for p in sorted((ROOT/'data_tambahan'/action).glob('*.mp4')):
        m=re.match(r'(person\d+)_',p.stem)
        if not m: raise ValueError('Nama harus memuat ID orang: '+p.name)
        extra.append(dict(path=str(p),video_id='local_'+p.stem,subject_id='local_'+m[1],action=action,label=CLASSES.index(action),split='train',source='local'))
if not any(r['action']=='idle' for r in extra):
    raise ValueError('Video idle tambahan diperlukan untuk eksperimen tujuh kelas.')
videos['source']='kth'
videos=pd.concat([videos,pd.DataFrame(extra)],ignore_index=True)
display(videos.groupby(['source','action','split']).size().rename('videos').reset_index())
print('Jogging tambahan boleh kosong; jogging KTH tetap digunakan.')
print('PERINGATAN: idle hanya memiliki data train. Tidak ada evaluasi independen idle.')

def prepare_image(frame):
    gray=cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)
    rgb=cv2.cvtColor(gray,cv2.COLOR_GRAY2RGB)
    h,w=rgb.shape[:2];ratio=min(IMAGE_SIZE/w,IMAGE_SIZE/h)
    nw,nh=max(1,round(w*ratio)),max(1,round(h*ratio))
    resized=cv2.resize(rgb,(nw,nh),interpolation=cv2.INTER_AREA)
    image=np.zeros((IMAGE_SIZE,IMAGE_SIZE,3),dtype=np.uint8)
    x=(IMAGE_SIZE-nw)//2;y=(IMAGE_SIZE-nh)//2
    image[y:y+nh,x:x+nw]=resized
    return image

def read_sequence(path,start,end):
    indices=np.linspace(int(start),int(end),FRAMES).round().astype(int)
    targets=set(indices.tolist());selected={}
    cap=cv2.VideoCapture(str(path))
    try:
        if not cap.isOpened(): raise ValueError('Video gagal dibuka: '+str(path))
        for i in range(int(end)+1):
            ok,frame=cap.read()
            if not ok or frame is None: raise ValueError(f'Decode berhenti frame {i}: {path}')
            if i in targets:selected[i]=prepare_image(frame)
    finally:cap.release()
    return np.stack([selected[int(i)] for i in indices]).astype(np.float32)

sequences=[]
for r in videos.to_dict('records'):
    cap=cv2.VideoCapture(r['path']);fps=float(cap.get(cv2.CAP_PROP_FPS));n=int(cap.get(cv2.CAP_PROP_FRAME_COUNT));cap.release()
    if not np.isfinite(fps) or fps<=0: raise ValueError('FPS tidak valid: '+r['path'])
    length=round(WINDOW_SECONDS*fps);stride=max(1,round(fps))
    if n<length: raise ValueError('Video lebih pendek dari jendela: '+r['path'])
    starts=list(range(0,n-length+1,stride))
    if starts[-1]!=n-length:starts.append(n-length)
    for i,start in enumerate(starts):
        sequences.append({**r,'sequence_id':r['video_id']+f'_window{i}','start':start,'end':start+length-1,'fps':fps})
sequences=pd.DataFrame(sequences)
sequences.to_csv(OUT/'sequence_manifest.csv',index=False)
display(pd.crosstab(sequences.action,sequences.split))
fig,axes=plt.subplots(2,4,figsize=(14,7))
for ax,action in zip(axes.ravel(),CLASSES):
    r=sequences[sequences.action==action].iloc[0]
    ax.imshow(read_sequence(r.path,r.start,r.end)[FRAMES//2].astype('uint8'));ax.set_title(action);ax.axis('off')
axes.ravel()[-1].axis('off');plt.tight_layout();plt.show()


## Ekstraksi fitur CNN
MobileNetV2 dibekukan dan berjalan dalam mode inferensi. Setiap subsekuens menjadi 16 vektor fitur berurutan. Cache memiliki kunci berdasarkan konfigurasi dan berkas sumber. Tidak ada pelatihan atau normalisasi yang menggunakan test.

In [ ]:
encoder=keras.applications.MobileNetV2(input_shape=(IMAGE_SIZE,IMAGE_SIZE,3),include_top=False,weights='imagenet',pooling='avg')
encoder.trainable=False
ENCODER_CONFIG={'name':'MobileNetV2','weights':'imagenet','image_size':IMAGE_SIZE,'frames':FRAMES,'gray':True,'resize':'letterbox','window_seconds':WINDOW_SECONDS,'dataset':'kth_plus_local_v1','backend':'torch','keras':keras.__version__,'torch':torch.__version__}
def encode_frames(frames):
    x=keras.applications.mobilenet_v2.preprocess_input(np.asarray(frames,dtype=np.float32))
    return keras.ops.convert_to_numpy(encoder(x,training=False)).astype('float32')

features=[];valid_rows=[];failures=[]
for r in tqdm(sequences.to_dict('records'),desc='Ekstraksi CNN'):
    stat=Path(r['path']).stat()
    key=hashlib.sha256(json.dumps({**ENCODER_CONFIG,'id':r['sequence_id'],'start':r['start'],'end':r['end'],'size':stat.st_size,'mtime':stat.st_mtime_ns},sort_keys=True).encode()).hexdigest()[:24]
    file=CACHE/(key+'.npy')
    if file.exists(): f=np.load(file,allow_pickle=False)
    else:
        try:
            frames=read_sequence(r['path'],r['start'],r['end'])
        except ValueError as exc:
            failures.append({**r,'error':str(exc)})
            continue
        f=encode_frames(frames)
        np.save(file,f,allow_pickle=False)
    if f.shape!=(FRAMES,1280) or not np.isfinite(f).all(): raise ValueError('Cache fitur tidak valid')
    features.append(f)
    valid_rows.append(r)
pd.DataFrame(failures,columns=list(sequences.columns)+['error']).to_csv(OUT/'decode_failures.csv',index=False)
if failures:
    print(f'PERHATIAN: {len(failures)} dari {len(sequences)} subsekuens gagal didekode dan dikecualikan. Detail: decode_failures.csv')
    display(pd.DataFrame(failures)[['video_id','sequence_id','split','error']])
if not valid_rows or len(failures)/len(sequences)>0.05:
    raise RuntimeError('Lebih dari 5% subsekuens gagal atau tidak ada data valid; periksa dataset sebelum training.')
sequences=pd.DataFrame(valid_rows).reset_index(drop=True)
for split in ['train','val','test']:
    if sequences[sequences.split==split].label.nunique()!=(len(CLASSES) if split=='train' else len(KTH_CLASSES)):
        raise RuntimeError(f'Kelas tidak lengkap setelah validasi decode: {split}')
sequences.to_csv(OUT/'sequence_manifest_valid.csv',index=False)
X=np.stack(features);y=sequences.label.to_numpy(dtype='int32')
masks={s:(sequences.split.to_numpy()==s) for s in ['train','val','test']}
print('Bentuk data:',X.shape)


## Training model sendiri
Bandingkan ANN yang merata-ratakan fitur frame dengan LSTM yang mempelajari urutannya. Pilih model dengan validation loss, kemudian evaluasi test sekali setelah pilihan dibekukan. Skor test tidak digunakan untuk memilih model.

In [ ]:
from keras import layers
from sklearn.utils.class_weight import compute_class_weight
weights=compute_class_weight(class_weight='balanced',classes=np.arange(len(CLASSES)),y=y[masks['train']])
class_weights=dict(enumerate(weights))
histories={};val_losses={}
for kind in ['ann','lstm']:
    keras.utils.set_random_seed(SEED)
    inp=layers.Input((FRAMES,1280))
    z=layers.GlobalAveragePooling1D()(inp) if kind=='ann' else layers.LSTM(64)(inp)
    z=layers.Dropout(0.3)(z)
    z=layers.Dense(32,activation='relu')(z)
    out=layers.Dense(len(CLASSES),activation='softmax')(z)
    model=keras.Model(inp,out,name=kind)
    model.compile(optimizer=keras.optimizers.Adam(1e-3),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss',patience=6,restore_best_weights=True),keras.callbacks.ModelCheckpoint(str(OUT/f'{kind}_best.keras'),monitor='val_loss',save_best_only=True)]
    h=model.fit(X[masks['train']],y[masks['train']],validation_data=(X[masks['val']],y[masks['val']]),epochs=EPOCHS,batch_size=BATCH_SIZE,class_weight=class_weights,callbacks=callbacks,verbose=2)
    histories[kind]=h.history
    val_losses[kind]=float(model.evaluate(X[masks['val']],y[masks['val']],verbose=0)[0])
BEST=min(val_losses,key=val_losses.get)
best_model=keras.models.load_model(OUT/f'{BEST}_best.keras')
best_model.save(OUT/'activity_model.keras')
config={**ENCODER_CONFIG,'classes':CLASSES,'selected_model':BEST,'seed':SEED,'window_seconds':WINDOW_SECONDS,'predict_every_seconds':PREDICT_EVERY_SECONDS,'audio':'removed','evaluation_unit':'official subsequence'}
(OUT/'pipeline_config.json').write_text(json.dumps(config,indent=2))
(OUT/'training_history.json').write_text(json.dumps(histories))
print('Model terpilih dari validation loss:',BEST,val_losses)
for kind,h in histories.items():
    plt.plot(h['loss'],label=kind+' train');plt.plot(h['val_loss'],label=kind+' val',linestyle='--')
plt.xlabel('Epoch');plt.ylabel('Loss');plt.legend();plt.tight_layout()
plt.savefig(OUT/'training_loss.png',dpi=160);plt.show()


In [ ]:
prob=best_model.predict(X[masks['test']],batch_size=BATCH_SIZE,verbose=0)
pred=prob.argmax(axis=1);truth=y[masks['test']]
report=classification_report(truth,pred,labels=np.arange(len(CLASSES)),target_names=CLASSES,output_dict=True,zero_division=0)
print(classification_report(truth,pred,labels=np.arange(len(CLASSES)),target_names=CLASSES,zero_division=0))
metrics={'test_scope':'KTH six classes only; idle and personal videos have no independent test','accuracy':float(accuracy_score(truth,pred)),'macro_f1':float(f1_score(truth,pred,labels=np.arange(len(KTH_CLASSES)),average='macro',zero_division=0)),'classification_report':report,'validation_losses':val_losses,'selected_model':BEST}
(OUT/'test_metrics.json').write_text(json.dumps(metrics,indent=2))
cm=confusion_matrix(truth,pred,labels=np.arange(len(CLASSES)))
ConfusionMatrixDisplay(cm,display_labels=CLASSES).plot(xticks_rotation=45)
plt.tight_layout();plt.savefig(OUT/'confusion_matrix.png',dpi=160);plt.show()
results=sequences[masks['test']].copy()
results['prediction']=[CLASSES[i] for i in pred]
results['model_score']=prob.max(axis=1)
results.to_csv(OUT/'test_predictions.csv',index=False)
display(results[results.action!=results.prediction][['video_id','action','prediction','model_score']].head(15))
print('Evaluasi hanya enam kelas KTH pada jendela dua detik. Jendela bertumpang tindih tidak independen. Tidak ada pengujian idle atau video pribadi terpisah; jangan menyimpulkan adaptasi berhasil dari data train.')


## Inferensi eksperimen adaptasi
Video di inference/video_uji.mp4 hanya menjadi demonstrasi. Pastikan berbeda dari seluruh rekaman adaptasi untuk pengujian independen. Model tujuh kelas belum mengenali seluruh aktivitas di luar daftar.


In [ ]:
def load_pipeline():
    cfg=json.loads((OUT/'pipeline_config.json').read_text())
    classifier=keras.models.load_model(OUT/'activity_model.keras')
    feature_model=keras.applications.MobileNetV2(input_shape=(cfg['image_size'],cfg['image_size'],3),include_top=False,weights=cfg['weights'],pooling='avg')
    feature_model.trainable=False
    return cfg,classifier,feature_model

cfg,classifier,feature_model=load_pipeline()
def inference_image(frame):
    size=cfg['image_size']
    gray=cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)
    rgb=cv2.cvtColor(gray,cv2.COLOR_GRAY2RGB)
    h,w=rgb.shape[:2];ratio=min(size/w,size/h)
    nw,nh=max(1,round(w*ratio)),max(1,round(h*ratio))
    image=np.zeros((size,size,3),dtype=np.uint8)
    image[(size-nh)//2:(size-nh)//2+nh,(size-nw)//2:(size-nw)//2+nw]=cv2.resize(rgb,(nw,nh),interpolation=cv2.INTER_AREA)
    return image

def run_inference(video_path):
    video_path=Path(video_path)
    if not video_path.exists(): raise FileNotFoundError(f'Unggah video ke {video_path}')
    local_input=(ROOT/'runtime')/('activity_input'+video_path.suffix)
    shutil.copy2(video_path,local_input)
    cap=cv2.VideoCapture(str(local_input))
    fps=float(cap.get(cv2.CAP_PROP_FPS));n=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH));height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    if not np.isfinite(fps) or fps<=0 or min(n,width,height)<=0:
        cap.release();raise ValueError('Metadata video tidak valid')
    # File hasil memakai FPS asli; frame tetap ditulis seluruhnya.
    temp=(ROOT/'runtime'/'activity_annotated.mp4')
    writer=cv2.VideoWriter(str(temp),cv2.VideoWriter_fourcc(*'mp4v'),fps,(width,height))
    if not writer.isOpened(): cap.release();raise RuntimeError('VideoWriter gagal')
    win=cfg['window_seconds'];step=cfg['predict_every_seconds']
    buffer=deque();records=[];last_sample=-float('inf');last_prediction=-float('inf')
    status='Mengumpulkan data';score=None
    try:
        for i in tqdm(range(n),desc='Inferensi'):
            ok,frame=cap.read()
            if not ok: raise ValueError(f'Frame rusak pada indeks {i}')
            t=i/fps  # Video CFR; VFR perlu dikonversi ke CFR terlebih dahulu.
            if t-last_sample>=1/10-1e-8:
                buffer.append((t,inference_image(frame)));last_sample=t
            while buffer and buffer[0][0]<t-win: buffer.popleft()
            if t>=win and t-last_prediction>=step-1e-8 and len(buffer)>=2:
                timestamps=np.array([b[0] for b in buffer])
                targets=np.linspace(t-win,t,cfg['frames'])
                indices=np.abs(timestamps[:,None]-targets[None,:]).argmin(axis=0)
                images=np.stack([buffer[j][1] for j in indices]).astype('float32')
                images=keras.applications.mobilenet_v2.preprocess_input(images)
                f=keras.ops.convert_to_numpy(feature_model(images,training=False))
                p=keras.ops.convert_to_numpy(classifier(f[None],training=False))[0]
                status=cfg['classes'][int(p.argmax())];score=float(p.max());last_prediction=t
            label=status if score is None else f'{status} | skor model {score:.2f}'
            scale=max(.4,min(width/900,1.2))
            cv2.putText(frame,label,(10,max(22,int(35*scale))),cv2.FONT_HERSHEY_SIMPLEX,scale,(0,255,255),max(1,int(2*scale)))
            writer.write(frame)
            records.append({'frame':i,'time_sec':t,'activity':status,'model_score':score})
    finally: cap.release();writer.release()
    output=OUT/'hasil_aktivitas.mp4'
    import imageio_ffmpeg
    ffmpeg=imageio_ffmpeg.get_ffmpeg_exe()
    if ffmpeg:
        subprocess.run([ffmpeg,'-y','-loglevel','error','-i',str(temp),'-an','-c:v','libx264','-pix_fmt','yuv420p','-movflags','+faststart',str(output)],check=True)
    else:
        shutil.copy2(temp,output)
        print('FFmpeg tidak tersedia; output mp4v mungkin tidak diputar pada semua browser.')
    pd.DataFrame(records).to_csv(OUT/'hasil_per_frame.csv',index=False)
    summary={'input':str(video_path),'frames':len(records),'fps':fps,'duration_seconds':len(records)/fps,'audio':'removed','timestamp_assumption':'constant frame rate','model':cfg['selected_model']}
    (OUT/'inference_summary.json').write_text(json.dumps(summary,indent=2))
    print('Hasil:',output)
    return output

# Jalankan setelah training atau setelah model tersimpan tersedia.
if INFERENCE_VIDEO.exists():
    result_video=run_inference(INFERENCE_VIDEO)
    from IPython.display import Video,display
    display(Video(str(result_video),embed=False))
else:
    print('Training selesai. Untuk inferensi, unggah:',INFERENCE_VIDEO)
    print('Kemudian jalankan kembali sel ini.')


## Bahan laporan
Gunakan hasil aktual dari `test_metrics.json`, grafik loss, confusion matrix, dan contoh video inferensi. Jelaskan pemisahan orang resmi, penggunaan CNN pralatih, training LSTM, dan keterbatasan perpindahan dari latar KTH ke video sendiri. Jangan mengklaim aplikasi real-time karena notebook memproses video offline.

File keluaran utama: `activity_model.keras`, `pipeline_config.json`, `training_history.json`, `test_metrics.json`, `test_predictions.csv`, `training_loss.png`, `confusion_matrix.png`, dan hasil inferensi jika video disediakan.